# Phase 3 — Exploratory Data Analysis (EDA)

This notebook connects to the **Hopsworks Feature Store**, retrieves the multi-city historical feature dataset (`aqi_features`, version 1), and performs EDA across all 8 configured Pakistani cities (**Lahore, Karachi, Islamabad, Faisalabad, Multan, Peshawar, Rawalpindi, Gujranwala**).

---

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Add project root to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from src.utils import hopsworks_login

print('Libraries imported successfully.')

In [ ]:
# Connect to Hopsworks Feature Store & fetch aqi_features v1
project = hopsworks_login()
fs = project.get_feature_store()
fg = fs.get_feature_group('aqi_features', version=1)

print('Reading dataset from Hopsworks Feature Store...')
df = fg.read()
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values(by=['city', 'timestamp']).reset_index(drop=True)

print(f'Successfully loaded dataset shape: {df.shape}')
print(f'Configured cities found: {df["city"].unique().tolist()}')
df.head()

## 1. Missing Data Report

Check missing percentage per column overall and broken down by city.

In [ ]:
# Overall missing values summary
missing_overall = df.isnull().mean() * 100
missing_summary = pd.DataFrame({'Missing (%)': missing_overall[missing_overall > 0].round(2)})
print('--- Overall Missing Value Percentage ---')
print(missing_summary if not missing_summary.empty else 'No missing values found overall.')

# Per-city missing values breakdown
city_missing = df.groupby('city').apply(lambda g: g.isnull().mean() * 100).round(2)
plt.figure(figsize=(12, 5))
sns.heatmap(city_missing.T, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': '% Missing'})
plt.title('Missing Data Percentage per Column by City')
plt.tight_layout()
plt.show()

## 2. Multi-City AQI Time Series Analysis

Visualization of historical AQI trends, seasonality, and diurnal cycles across cities.

In [ ]:
# Multi-city time series plot
plt.figure(figsize=(16, 7))
sns.lineplot(data=df, x='timestamp', y='aqi', hue='city', alpha=0.85, linewidth=1.5)
plt.title('Hourly Air Quality Index (AQI) Time Series by City', fontsize=14, fontweight='bold')
plt.xlabel('Date (UTC)')
plt.ylabel('AQI Value')
plt.legend(title='City', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Distribution of AQI (Overall & Per City)

Assessing distribution skewness, extreme values, and ranges across cities.

In [ ]:
# Overall AQI distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df['aqi'], kde=True, ax=axes[0], color='teal', bins=40)
axes[0].set_title('Overall AQI Distribution (All Cities)')
axes[0].set_xlabel('AQI Value')

# Per-city AQI distribution
sns.boxplot(data=df, x='city', y='aqi', ax=axes[1], palette='Set2')
axes[1].set_title('AQI Range & Boxplot per City')
axes[1].set_xlabel('City')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

Analyzing correlations between pollutants (PM2.5, PM10, O3, NO2, SO2, CO), lag features, and AQI.

In [ ]:
# Correlation matrix
numeric_cols = ['aqi', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'co', 'aqi_lag_1h', 'aqi_lag_24h', 'aqi_rolling_mean_24h', 'target_24h', 'target_48h', 'target_72h']
valid_num_cols = [c for c in numeric_cols if c in df.columns]
corr_matrix = df[valid_num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Feature & Pollutant Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Autocorrelation Analysis (ACF)

Evaluating temporal dependency and autocorrelation structure per city up to 72 lags.

In [ ]:
# Compute ACF up to 72 hours per city using pandas built-in .autocorr()
lags = 72
plt.figure(figsize=(14, 6))

for city_name in df['city'].unique():
    city_series = df[df['city'] == city_name]['aqi'].dropna()
    if len(city_series) > lags:
        acf_vals = [city_series.autocorr(lag=i) for i in range(lags + 1)]
        plt.plot(range(lags + 1), acf_vals, label=city_name, alpha=0.8, linewidth=2)

plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title('AQI Autocorrelation Function (ACF) per City (0-72 Hours)')
plt.xlabel('Lag (Hours)')
plt.ylabel('Autocorrelation')
plt.legend(title='City', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()